In [ ]:
import os
import random
import time
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from IPython.display import display

from sklearn.model_selection import GroupShuffleSplit, train_test_split
#BEGIN RH - add more metrics
from sklearn.metrics import (
    accuracy_score, 
    classification_report, 
    confusion_matrix,
    ConfusionMatrixDisplay, 
    log_loss, 
    precision_score, 
    recall_score, 
    f1_score,
)
#END RH

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

def seed_everything(seed=42):
    """
    Make results more reproducible.

    In deep learning, perfect reproducibility is not always guaranteed,
    especially on GPU, but this helps a lot.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

In [ ]:
ROOT = "/kaggle/input/competitions/state-farm-distracted-driver-detection"
TRAIN_DIR = "/kaggle/input/competitions/state-farm-distracted-driver-detection/imgs/train"
csv_path = ROOT +'/'+ "driver_imgs_list.csv"
df = pd.read_csv(csv_path)

In [ ]:
df

In [ ]:
CLASS_NAMES = {
    "c0": "safe driving",
    "c1": "texting - right",
    "c2": "talking on the phone - right",
    "c3": "texting - left",
    "c4": "talking on the phone - left",
    "c5": "operating the radio",
    "c6": "drinking",
    "c7": "reaching behind",
    "c8": "hair and makeup",
    "c9": "talking to passenger",
}

CLASS_ORDER = [f"c{i}" for i in range(10)]
NUM_CLASSES = len(CLASS_ORDER)

df["class_name"] = df["classname"].map(CLASS_NAMES)

In [ ]:
df

In [ ]:
df["path"] = df.apply(lambda row: str(TRAIN_DIR +'/'+ row["classname"] +'/'+ row["img"]), axis=1)

In [ ]:
df

In [ ]:
print("Number of images:", len(df))
print("Number of unique drivers:", df["subject"].nunique())
print("Number of classes:", df["classname"].nunique())

print("\nDuplicate rows:")
print(df.duplicated(subset=["subject", "classname", "img"]).sum())

class_counts = (
    df["classname"]
    .value_counts()
    .reindex(CLASS_ORDER)
    .rename_axis("class_id")
    .reset_index(name="count")
)

class_counts["class_name"] = class_counts["class_id"].map(CLASS_NAMES)
class_counts["percent"] = (class_counts["count"] / class_counts["count"].sum() * 100).round(2)

display(class_counts)

plt.figure(figsize=(10, 4))
plt.bar(class_counts["class_id"], class_counts["count"])
plt.xticks(rotation=0)
plt.xlabel("Class")
plt.ylabel("Number of images")
plt.title("Class distribution")
plt.show()

In [ ]:
subject_counts = (
    df["subject"]
    .value_counts()
    .rename_axis("subject")
    .reset_index(name="image_count")
)

print("Images per driver:")
display(subject_counts.describe().T)

plt.figure(figsize=(10, 4))
plt.bar(subject_counts["subject"], subject_counts["image_count"])
plt.xticks(rotation=90)
plt.ylabel("Number of images")
plt.title("Images per driver / subject")
plt.show()

print("\nFirst rows:")
display(subject_counts.head())

In [ ]:
# Check image dimensions on a sample.

sample_for_size = df.sample(min(500, len(df)), random_state=42)

sizes = []
for p in sample_for_size["path"]:
    with Image.open(p) as img:
        width, height = img.size
        sizes.append((width, height))

size_df = pd.DataFrame(sizes, columns=["width", "height"])

print("Image size summary from a sample:")
display(size_df.describe().T)

print("\nMost common sizes:")
display(size_df.value_counts().head(10).to_frame("count"))

In [ ]:
def show_samples_by_class(dataframe, n_per_class=2, seed=42):
    """
    Show a few images from each class.
    """
    rng = np.random.default_rng(seed)

    rows = []
    for class_id in CLASS_ORDER:
        class_df = dataframe[dataframe["classname"] == class_id]
        if len(class_df) == 0:
            continue

        take_n = min(n_per_class, len(class_df))
        sampled_idx = rng.choice(class_df.index, size=take_n, replace=False)
        rows.extend(sampled_idx)

    n_images = len(rows)
    n_cols = n_per_class
    n_rows = int(np.ceil(n_images / n_cols))

    plt.figure(figsize=(4 * n_cols, 3 * n_rows))

    for i, idx in enumerate(rows):
        row = dataframe.loc[idx]
        img = Image.open(row["path"]).convert("RGB")

        ax = plt.subplot(n_rows, n_cols, i + 1)
        ax.imshow(img)
        ax.set_title(f"{row['classname']} - {row['class_name']}\nsubject={row['subject']}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

show_samples_by_class(df, n_per_class=3)

In [ ]:
def show_samples_by_subject(dataframe, n_per_subject=3, seed=42, max_subjects=None):
    """
    Show a few images from each subject / driver.
    """
    rng = np.random.default_rng(seed)

    # Make sure class_name exists
    dataframe = dataframe.copy()
    if "class_name" not in dataframe.columns:
        dataframe["class_name"] = dataframe["classname"].map(CLASS_NAMES)

    subjects = sorted(dataframe["subject"].unique())

    if max_subjects is not None:
        subjects = subjects[:max_subjects]

    rows = []

    for subject in subjects:
        subject_df = dataframe[dataframe["subject"] == subject]

        if len(subject_df) == 0:
            continue

        take_n = min(n_per_subject, len(subject_df))
        sampled_idx = rng.choice(subject_df.index, size=take_n, replace=False)
        rows.extend(sampled_idx)

    n_images = len(rows)
    n_cols = n_per_subject
    n_rows = int(np.ceil(n_images / n_cols))

    plt.figure(figsize=(4 * n_cols, 3 * n_rows))

    for i, idx in enumerate(rows):
        row = dataframe.loc[idx]
        img = Image.open(row["path"]).convert("RGB")

        ax = plt.subplot(n_rows, n_cols, i + 1)
        ax.imshow(img)
        ax.set_title(
            f"subject={row['subject']}\n"
            f"{row['classname']} - {row['class_name']}"
        )
        ax.axis("off")

    plt.tight_layout()
    plt.show()

show_samples_by_subject(df, n_per_subject=5, max_subjects=10)

In [ ]:
# Class distribution per subject

CLASS_ORDER = [f"c{i}" for i in range(10)]

# Make sure class_name exists
if "class_name" not in df.columns:
    df["class_name"] = df["classname"].map(CLASS_NAMES)

# Count images for each subject-class pair
subject_class_counts = pd.crosstab(
    df["subject"],
    df["classname"]
).reindex(columns=CLASS_ORDER, fill_value=0)

# Convert counts to percentages within each subject
subject_class_percent = (
    subject_class_counts
    .div(subject_class_counts.sum(axis=1), axis=0)
    .mul(100)
    .round(2)
)

subject_class_percent.plot(
    kind="bar",
    stacked=True,
    figsize=(14, 6),
    width=0.85
)

plt.axhline(100, linewidth=1)
plt.xlabel("Subject")
plt.ylabel("Percentage of images")
plt.title("Class composition per subject")
plt.legend(title="Class", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# normal random split

random_train_df, random_valid_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["classname"]
)

random_train_subjects = set(random_train_df["subject"])
random_valid_subjects = set(random_valid_df["subject"])
random_overlap = random_train_subjects.intersection(random_valid_subjects)

print("RANDOM SPLIT")
print("Train images:", len(random_train_df))
print("Valid images:", len(random_valid_df))
print("Unique train drivers:", len(random_train_subjects))
print("Unique valid drivers:", len(random_valid_subjects))
print("Drivers appearing in BOTH train and validation:", len(random_overlap))
print("Example overlapping drivers:", sorted(list(random_overlap))[:10])

In [ ]:
# group split by driver / subject

#BEGIN RH
best = None
for seed in range(200):
    #Split 70-30
    groupShuffle = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=seed)
    trainingIndex, testingIndex = next(groupShuffle.split(df, groups=df["subject"]))
    frac = len(testingIndex) / len(df)

    #every class for every set
    allow = (df.iloc[trainingIndex]["classname"].nunique() == NUM_CLASSES and df.iloc[testingIndex]["classname"].nunique() == NUM_CLASSES)
    if allow and (best is None or abs(frac - 0.3) < best[0]):
        best = (abs(frac- 0.3), seed, trainingIndex, testingIndex)

_, SPLIT_SEED, trainIndex, testIndex = best
trainDf = df.iloc[trainIndex].reset_index(drop=True)
testDf = df.iloc[testIndex].reset_index(drop=True)

assert set(trainDf["subject"]).isdisjoint(testDf["subject"])


print("GROUP SPLIT BY DRIVER")
print("Train images:", len(trainDf))
print("Split Seed: ", SPLIT_SEED)
print(f"Train: {len(trainDf)} imgs ({len(trainDf)/len(df):.1%}), {trainDf['subject'].nunique()} drivers ")
print(f"Test: {len(testDf)} imgs ({len(testDf)/len(df):.1%}), {testDf['subject'].nunique()} drivers ")

print("\nTrain class distribution:")
display(trainDf["classname"].value_counts().reindex(CLASS_ORDER).to_frame("train_count"))

#END RH


In [ ]:
#BEGIN RH - edit for updated split
MAX_TRAIN_IMAGES_PER_CLASS = None
MAX_VALID_IMAGES_PER_CLASS = None

def sample_per_class(dataframe, max_per_class=None, seed=42):
    """
    Keep at most max_per_class images for each class.
    """
    if max_per_class is None:
        return dataframe.sample(frac=1, random_state=seed).reset_index(drop=True)

    parts = []
    for class_id in CLASS_ORDER:
        class_df = dataframe[dataframe["classname"] == class_id]
        n = min(max_per_class, len(class_df))
        parts.append(class_df.sample(n=n, random_state=seed))

    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)
# random_train_df
train_df_small = sample_per_class(trainDf, MAX_TRAIN_IMAGES_PER_CLASS)
valid_df_small = sample_per_class(testDf, MAX_VALID_IMAGES_PER_CLASS)

#END RH

print("Training subset shape:", train_df_small.shape)
print("Validation subset shape:", valid_df_small.shape)

print("\nTraining subset class counts:")
display(train_df_small["classname"].value_counts().reindex(CLASS_ORDER).to_frame("count"))

print("\nValidation subset class counts:")
display(valid_df_small["classname"].value_counts().reindex(CLASS_ORDER).to_frame("count"))

In [ ]:
IMG_SIZE = 128

MEAN = [0.5, 0.5, 0.5]
STD = [0.5, 0.5, 0.5]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    # Small rotation can make model more robust.
    transforms.RandomRotation(degrees=5),

    # Lighting/color changes are realistic.
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.02
    ),

    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

valid_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

class_to_idx = {class_id: i for i, class_id in enumerate(CLASS_ORDER)}
idx_to_class = {i: class_id for class_id, i in class_to_idx.items()}

class DriverImageDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        # Open image
        image = Image.open(row["path"]).convert("RGB")

        # Transform image
        if self.transform is not None:
            image = self.transform(image)

        # Convert class id to integer label
        label = class_to_idx[row["classname"]]

        return image, label

train_dataset = DriverImageDataset(train_df_small, transform=train_transform)
valid_dataset = DriverImageDataset(valid_df_small, transform=valid_transform)


print("Train dataset length:", len(train_dataset))
print("Valid dataset length:", len(valid_dataset))

one_image, one_label = train_dataset[0]

print("\nOne image tensor shape:", one_image.shape)
print("One label:", one_label, "->", idx_to_class[one_label], "->", CLASS_NAMES[idx_to_class[one_label]])
print("Pixel min:", one_image.min().item())
print("Pixel mean:", one_image.mean().item())
print("Pixel max:", one_image.max().item())

In [ ]:
BATCH_SIZE = 64


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=torch.cuda.is_available()
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=torch.cuda.is_available()
)

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            # Input: [B, 3, 128, 128]
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            # [B, 32, 64, 64]

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            # [B, 64, 32, 32]

            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            # [B, 128, 16, 16]

            # Convert any remaining spatial size to 4x4
            nn.AdaptiveAvgPool2d((4, 4)),
            # [B, 128, 4, 4]
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.30),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(p=0.30),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SmallCNN(num_classes=NUM_CLASSES).to(device)

print(model)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {num_params:,}")

# Conv2d(64, 128, kernel_size=3)
# Έχει:
# 128 filters * (64 input channels * 3 * 3 + 1 bias) = 73,856 params

In [ ]:
def train_one_epoch(model, dataloader, loss_fn, optimizer, device):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for images, labels in dataloader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Forward
        outputs = model(images)
        loss = loss_fn(outputs, labels)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Metrics
        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        predictions = outputs.argmax(dim=1)
        total_correct += (predictions == labels).sum().item()
        total_examples += batch_size

    avg_loss = total_loss / total_examples
    accuracy = total_correct / total_examples

    return avg_loss, accuracy


def evaluate(model, dataloader, loss_fn, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    all_predictions = []
    all_labels = []
    all_probabilities = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = loss_fn(outputs, labels)

            probabilities = torch.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (predictions == labels).sum().item()
            total_examples += batch_size

            all_predictions.append(predictions.cpu())
            all_labels.append(labels.cpu())
            all_probabilities.append(probabilities.cpu())

    avg_loss = total_loss / total_examples
    accuracy = total_correct / total_examples

    all_predictions = torch.cat(all_predictions).numpy()
    all_labels = torch.cat(all_labels).numpy()
    all_probabilities = torch.cat(all_probabilities).numpy()

    return avg_loss, accuracy, all_predictions, all_labels, all_probabilities

#BEGIN RH - add additional meterics
def computeMetrics(yTrue, yPredict, yProb):
    labels = list(range(NUM_CLASSES))
    return {
        "cross entropy ": log_loss(yTrue, yProb, labels==labels),
        "accuracy ": accuracy_score(yTrue, yPredict),
        "precision ": precision_score(yTrue, yPredict, average="macro", zero_division=0),
        "recall ": recall_score(yTrue, yPredict, average="macro", zero_division=0),
        "F1 ": f1_score(yTrue, yPredict, average="macro", zero_division=0),
    }
#END - RH

In [ ]:
loss_fn = nn.CrossEntropyLoss()

# If classes were very imbalanced, we could add class weights.

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 3

history = []

for epoch in range(EPOCHS):
    start = time.time()

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        loss_fn,
        optimizer,
        device
    )

    valid_loss, valid_acc, valid_pred, valid_true, valid_probs = evaluate(
        model,
        valid_loader,
        loss_fn,
        device
    )

#BEGIN - RH - update training to include additional meterics
    epoch_time = time.time() - start
    m = computeMetrics(valid_true, valid_pred, valid_probs)

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "valid_loss": valid_loss,
        "valid_acc": valid_acc,
        "time_sec": epoch_time,
        "test_loss ": m["cross_entropy"],
        "test_acc ": m["accuracy"],
        "test_precision ": m["precision"],
        "test_f1 ": m["f1"],
    })

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"train_acc={train_acc:.4f} | "
        f"valid_loss={valid_loss:.4f} | "
        f"valid_acc={valid_acc:.4f} | "
        f"time={epoch_time:.1f}s | "
        f"test_cross_entropy={m['cross_entropy']:.4f} | "
        f"test_acc={m['accuracy']:.4f} | "
        f"test_predict={m['precision']:.4f} | "
        f"test_f1={m['f1']:.4f}"
    )
#END - RH

In [ ]:
history_df = pd.DataFrame(history)
display(history_df)

plt.figure(figsize=(7, 4))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train loss")
plt.plot(history_df["epoch"], history_df["valid_loss"], marker="o", label="valid loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs validation loss")
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history_df["epoch"], history_df["train_acc"], marker="o", label="train acc")
plt.plot(history_df["epoch"], history_df["valid_acc"], marker="o", label="valid acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs validation accuracy")
plt.legend()
plt.show()

In [ ]:
# Run final evaluation again, to make sure variables are fresh.
valid_loss, valid_acc, valid_pred, valid_true, valid_probs = evaluate(
    model,
    valid_loader,
    loss_fn,
    device
)

#BEGIN - RH - update final metrics 
final = computeMetrics(valid_true, valid_pred, valid_probs)
for k,v in final.items():
    print(f"{k:>14}: {v:.4f}")

print(f"Validation loss: {valid_loss:.4f}")
print(f"Validation accuracy: {valid_acc:.4f}")

target_names = [f"{class_id}: {CLASS_NAMES[class_id]}" for class_id in CLASS_ORDER]

print("\nClassification report:")
print(classification_report(valid_true, valid_pred, target_names=target_names, zero_division=0))
#END - RH

In [ ]:
cm = confusion_matrix(valid_true, valid_pred, labels=list(range(NUM_CLASSES)))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=CLASS_ORDER
)

fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, values_format="d", xticks_rotation=45)
plt.title("Validation confusion matrix")
plt.show()